In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
import random

new_txt_file = "C:/Users/91892/Downloads/amazon_test.txt"
# Read URLs from file
with open(new_txt_file, "r") as file:
    product_urls = [line.strip() for line in file.readlines()]

# Setup Selenium WebDriver with Headless Mode
chrome_options = Options()
chrome_options.add_argument("--headless=new")  # Run in headless mode
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--window-size=1920x1080")  # Ensure full page load
chrome_options.add_argument("--disable-blink-features=AutomationControlled")  # Avoid bot detection

# Path to your chromedriver.exe
service = Service("chromedriver.exe")  
driver = webdriver.Chrome(options=chrome_options)

# Function to extract product specifications
def get_product_specifications(soup):
    specs = {}

    # Try to extract from "Technical Details" section
    tech_table = soup.find("table", {"id": "productDetails_techSpec_section_1"})
    if tech_table:
        rows = tech_table.find_all("tr")
        for row in rows:
            key = row.find("th").get_text(strip=True) if row.find("th") else None
            value = row.find("td").get_text(strip=True) if row.find("td") else None
            if key and value:
                specs[key] = value

    # Try to extract from "Additional Details" section
    additional_table = soup.find("table", {"id": "productDetails_detailBullets_sections1"})
    if additional_table:
        rows = additional_table.find_all("tr")
        for row in rows:
            key = row.find("th").get_text(strip=True) if row.find("th") else None
            value = row.find("td").get_text(strip=True) if row.find("td") else None
            if key and value:
                specs[key] = value

    return specs

# Function to scrape Amazon product details
def scrape_amazon_product(url):
    print(f"🔍 Scraping: {url}")
    driver.get(url)
    
    try:
        # Wait for the product title to load
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "productTitle"))
        )
    except:
        print(f"⚠️ Skipping {url} - Page did not load properly")
        return None

    time.sleep(5)  # Additional sleep for stability

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # Check if product is in stock
    availability = soup.find("div", {"id": "availability"})
    in_stock_text = availability.get_text(strip=True) if availability else "Out of Stock"

    actual_price = None
    discounted_price = None

    # Extract prices only if the product is in stock
    if "unavailable" not in in_stock_text:
        actual_price_element = soup.find("span", class_="a-price a-text-price")
        actual_price = (
            actual_price_element.find("span", class_="a-offscreen").get_text(strip=True)
            if actual_price_element and actual_price_element.find("span", class_="a-offscreen")
            else None
        )

        # Extract Discounted Price (Final Price)
        discounted_price_whole = soup.find("span", class_="a-price-whole")
        discounted_price_fraction = soup.find("span", class_="a-price-fraction")

        if discounted_price_whole and discounted_price_fraction:
            discounted_price = f"{discounted_price_whole.get_text(strip=True)}.{discounted_price_fraction.get_text(strip=True)}"
        elif discounted_price_whole:
            discounted_price = discounted_price_whole.get_text(strip=True)

    time.sleep(random.randint(2, 5))
    # Extract product details
    data = {
        "Product Name": soup.find("span", id="productTitle").get_text(strip=True) if soup.find("span", id="productTitle") else None,
        "Brand": soup.find("a", id="bylineInfo").get_text(strip=True) if soup.find("a", id="bylineInfo") else None,
        "Seller": soup.find("a", {"id": "sellerProfileTriggerId"}).get_text(strip=True) if soup.find("a", {"id": "sellerProfileTriggerId"}) else None,
        "Specifications": get_product_specifications(soup),  # Extracted specifications
        "Actual Price": actual_price,
        "Discounted Price": discounted_price,
        "In Stock": in_stock_text,
        "Review Count": soup.find("span", id="acrCustomerReviewText").get_text(strip=True) if soup.find("span", id="acrCustomerReviewText") else None,
        "Ratings": soup.find("span", class_="a-icon-alt").get_text(strip=True) if soup.find("span", class_="a-icon-alt") else None,
        'URL': url
    }

    return data

# Scrape all product URLs
all_products = []
for url in product_urls:
    product_data = scrape_amazon_product(url)
    if product_data:  # Only append if data was successfully retrieved
        all_products.append(product_data)

# Convert list of dictionaries to a DataFrame
df = pd.DataFrame(all_products)

# Save as CSV
csv_path = "C:/Users/91892/Downloads/amazon_products_20k.csv"
df.to_csv(csv_path, index=False, encoding="utf-8")
print(f"✅ Scraped data saved as {csv_path}")

# Close the browser
driver.quit()


🔍 Scraping: https://www.amazon.in/MAYCREATE%C2%AE-Pencil-Styling-Mustache-Eyebrows/dp/B0BGLT5XFJ/ref=sr_1_386?qid=1679215835&s=hpc&sr=1-386
🔍 Scraping: https://www.amazon.in/Tash-Hair-Automatic-Rechargeable-Adjustable/dp/B0BD8XJK5R/ref=sr_1_389?qid=1679215835&s=hpc&sr=1-389
🔍 Scraping: https://www.amazon.in/VGR-V-596-Professional-Electric-Curling/dp/B0B4B9973W/ref=sr_1_385?qid=1679215835&s=hpc&sr=1-385
🔍 Scraping: https://www.amazon.in/BARBER-Professional-Stylish-Dryers-Womens/dp/B08P493PBC/ref=sr_1_388?qid=1679215835&s=hpc&sr=1-388
🔍 Scraping: https://www.amazon.in/Electric-Straightener-Straightening-Detangling-functional/dp/B09J2TJPQB/ref=sr_1_390?qid=1679215835&s=hpc&sr=1-390
🔍 Scraping: https://www.amazon.in/Colgate-Advanced-Toothbrush-intensities-Convenient/dp/B0B2PLSN6X/ref=sr_1_391?qid=1679215835&s=hpc&sr=1-391
🔍 Scraping: https://www.amazon.in/LetsShave-setting-Charging-Cordless-LSER-2/dp/B098P5JN7J/ref=sr_1_392?qid=1679215835&s=hpc&sr=1-392
🔍 Scraping: https://www.amazon.in/AG

In [2]:
df

,Product Name,Brand,Seller,Specifications,Actual Price,Discounted Price,In Stock,Review Count,Ratings,URL
0,MAYCREATE® Beard Pencil Filler for Men Dual Ti...,Visit the MAYCREATE Store,None,"{'Manufacturer': 'MAYCREATE, cs.service01@outl...",None,None,Currently unavailable.We don't know when or if...,46 ratings,3.3 out of 5 stars,https://www.amazon.in/MAYCREATE%C2%AE-Pencil-S...
1,Tash Hair Cordless Automatic Curler - Professi...,Visit the Tash Hair Store,None,"{'Manufacturer': 'Tash Hair_ Retail', 'Item pa...",None,None,Currently unavailable.We don't know when or if...,14 ratings,3.6 out of 5 stars,https://www.amazon.in/Tash-Hair-Automatic-Rech...
2,VGR V-596 Professional Electric Hair Curling W...,None,Electronics Bazaar Store,{},"₹1,380",967.00,In stock,160 ratings,4.0 out of 5 stars,https://www.amazon.in/VGR-V-596-Professional-E...
3,BARBER Professional Stylish Hair Dryers for Wo...,Visit the BARBER Store,ThaBeautystore,"{'Manufacturer': 'BARBER', 'Country of Origin'...",₹999,649..00,In stock,341 ratings,4.1 out of 5 stars,https://www.amazon.in/BARBER-Professional-Styl...
4,"FEXMY? Electric Hair Straightener Brush,Men Qu...",Visit the FEXMY Store,FEXMY®,"{'Country of Origin': '‎India', 'Item part num...","₹1,599",469..00,In stock,27 ratings,3.2 out of 5 stars,https://www.amazon.in/Electric-Straightener-St...
5,Colgate Advanced Electric Toothbrush for adult...,Brand: Colgate,None,{'Manufacturer': 'Ningbo Seago Electric Co. Lt...,None,None,Currently unavailable.We don't know when or if...,90 ratings,3.1 out of 5 stars,https://www.amazon.in/Colgate-Advanced-Toothbr...
6,LetsShave All Under Trimmer For Men (All Under...,Visit the LetsShave Store,None,"{'Manufacturer': 'ICARE Electric, ICARE Electr...",None,None,Currently unavailable.We don't know when or if...,23 ratings,3.6 out of 5 stars,https://www.amazon.in/LetsShave-setting-Chargi...
7,"AGARO Hair Straightener, Ceramic Coated Plates...",Visit the AGARO Store,Electronics Bazaar Store,{'Manufacturer': 'Universal Corporation Limite...,"₹2,499",924..00,In stock,66 ratings,3.8 out of 5 stars,https://www.amazon.in/AGARO-Straightener-Adjus...
8,Ikonic Ultralight Professional Hair Dryer 2000...,Visit the IKONIC Store,SSIZ INTERNATIONAL PVT LTD,"{'Manufacturer': 'SSIZ INTERNATIONAL PVT LTD, ...","₹2,300","1,496..00",In stock,133 ratings,3.9 out of 5 stars,https://www.amazon.in/Ikonic-Ultralight-2000-D...
9,Philips 1000 Watts HP8143/00 Hair Dryer (Pink),Visit the PHILIPS Store,Electronics_Bazaar Store,"{'Manufacturer': 'Philips', 'Item model number...","₹1,395","1,390..00",Only 1 left in stock.,922 ratings,4.2 out of 5 stars,https://www.amazon.in/Philips-HP8143-00-Hair-D...
